# Análise exploratória — Assistência Técnica de Celulares

Análise da base de ordens de serviço (OS) já limpa (`dados/ordens_servico_limpo.csv`).

> Os dados são **simulados** com base em informações passadas pela empresa.

In [1]:
import pandas as pd
import plotly.express as px

# Lê a base limpa, já convertendo as colunas de data
df = pd.read_csv(
    "dados/ordens_servico_limpo.csv",
    sep=";",
    parse_dates=["data_entrada", "data_conclusao", "data_entrega"],
)

# OS entregues: usadas para receita e taxa de retorno
entregues = df[df["status"] == "Entregue"]

## 1. Visão geral da base

In [2]:
print("Total de OS:", len(df))
print("Período:", df["data_entrada"].min().date(), "a", df["data_entrada"].max().date())
print("\nOS por status:")
print(df["status"].value_counts())
df.head()

Total de OS: 2478
Período: 2025-11-01 a 2026-10-17

OS por status:
status
Entregue           2328
Cancelado           123
Em andamento         14
Pronto               12
Aguardando peça       1
Name: count, dtype: int64


,os_id,data_entrada,data_conclusao,data_entrega,marca,modelo,defeito,tipo_reparo,tecnico,status,valor,retorno_garantia,dias_reparo,mes
0,OS-0001,2025-11-01,2025-11-02,2025-11-06,Apple,iPhone 11,Software travando,Reinstalação de software,Juliana,Entregue,96.59,Não,1.0,2025-11
1,OS-0002,2025-11-01,2025-11-04,2025-11-07,Samsung,Galaxy A54,Tela quebrada,Troca de tela,Marcos,Entregue,332.82,Não,3.0,2025-11
2,OS-0003,2025-11-01,2025-11-02,2025-11-02,Samsung,Galaxy M34,Tela quebrada,Troca de tela,Carlos,Entregue,573.51,Não,1.0,2025-11
3,OS-0004,2025-11-01,2025-11-02,2025-11-04,Apple,iPhone 12,Tela quebrada,Troca de tela,Marcos,Entregue,876.34,Não,1.0,2025-11
4,OS-0005,2025-11-01,2025-11-02,2025-11-03,Samsung,Galaxy M34,Conector de carga,Troca de conector,Rafael,Entregue,124.35,Não,1.0,2025-11


In [3]:
fig = px.pie(df, names="status", title="OS por status", hole=0.4)
fig.show()

A base tem cerca de 2,5 mil OS em quase 12 meses. A grande maioria já foi **entregue**;
cerca de 5% foram canceladas e poucas estão em andamento (só as que chegaram nos últimos dias).

## 2. Aparelhos recebidos por mês

In [4]:
por_mes = df.groupby("mes").size().reset_index(name="aparelhos")

fig = px.bar(por_mes, x="mes", y="aparelhos", title="Aparelhos recebidos por mês", text="aparelhos")
fig.update_layout(xaxis_title="Mês", yaxis_title="Aparelhos")
fig.show()

In [5]:
# Compara dezembro/janeiro com a média dos outros meses completos (fev a set)
pico = por_mes[por_mes["mes"].isin(["2025-12", "2026-01"])]["aparelhos"].mean()
normal = por_mes[(por_mes["mes"] >= "2026-02") & (por_mes["mes"] <= "2026-09")]["aparelhos"].mean()
print(f"Média dez/jan: {pico:.0f} | Média fev-set: {normal:.0f} | Diferença: {pico / normal - 1:.0%}")

Média dez/jan: 267 | Média fev-set: 207 | Diferença: 29%


Existe **pico em dezembro e janeiro** (fim de ano e férias), com bem mais aparelhos que a média
dos outros meses. Nos demais meses o movimento é estável, perto de 200 aparelhos por mês.
Outubro aparece baixo só porque a base vai até 17/10.

## 3. Defeitos mais frequentes

In [6]:
defeitos = df["defeito"].value_counts().reset_index()
defeitos.columns = ["defeito", "quantidade"]
defeitos["percentual"] = (defeitos["quantidade"] / len(df) * 100).round(1)

fig = px.bar(defeitos, x="quantidade", y="defeito", orientation="h",
             title="Defeitos mais frequentes", text="percentual")
fig.update_traces(texttemplate="%{text}%")
fig.update_layout(yaxis={"categoryorder": "total ascending"}, xaxis_title="OS", yaxis_title="")
fig.show()
defeitos

,defeito,quantidade,percentual
0,Tela quebrada,883,35.6
1,Bateria viciada,541,21.8
2,Conector de carga,359,14.5
3,Software travando,243,9.8
4,Danos por água,240,9.7
5,Não liga,212,8.6


**Tela quebrada** é de longe o defeito mais comum (mais de um terço das OS), seguida de
bateria viciada. Juntos, os dois somam mais da metade dos atendimentos.

## 4. Tempo médio de reparo por tipo de reparo

In [7]:
tempo = df.groupby("tipo_reparo")["dias_reparo"].mean().round(1).reset_index()
tempo = tempo.sort_values("dias_reparo")

fig = px.bar(tempo, x="dias_reparo", y="tipo_reparo", orientation="h",
             title="Tempo médio de reparo (dias)", text="dias_reparo")
fig.update_layout(xaxis_title="Dias", yaxis_title="")
fig.show()
print(f"Tempo médio geral: {df['dias_reparo'].mean():.1f} dias")

Tempo médio geral: 2.6 dias


**Reparo de placa** e **limpeza de oxidação** são os mais demorados (5 a 6 dias em média).
Software, bateria e conector ficam prontos em 1 a 2 dias. A troca de tela às vezes atrasa
porque fica esperando peça.

## 5. Taxa de retorno na garantia

In [8]:
# Taxa de retorno = OS que voltaram na garantia / OS entregues
def taxa_retorno(coluna):
    taxa = entregues.groupby(coluna)["retorno_garantia"].apply(lambda s: (s == "Sim").mean() * 100)
    return taxa.round(1).sort_values().reset_index(name="taxa_retorno")

print(f"Taxa de retorno geral: {(entregues['retorno_garantia'] == 'Sim').mean():.1%}")

retorno_defeito = taxa_retorno("defeito")
fig = px.bar(retorno_defeito, x="taxa_retorno", y="defeito", orientation="h",
             title="Taxa de retorno por defeito (%)", text="taxa_retorno")
fig.update_layout(xaxis_title="% de retorno", yaxis_title="")
fig.show()

Taxa de retorno geral: 10.0%


In [9]:
retorno_tecnico = taxa_retorno("tecnico")
fig = px.bar(retorno_tecnico, x="taxa_retorno", y="tecnico", orientation="h",
             title="Taxa de retorno por técnico (%)", text="taxa_retorno")
fig.update_layout(xaxis_title="% de retorno", yaxis_title="")
fig.show()

**Danos por água** é o defeito que mais volta (cerca de 30% dos aparelhos), seguido de
"Não liga" (reparo de placa). Entre os técnicos, **Marcos** (o técnico mais novo) tem uma taxa
de retorno de mais que o dobro dos colegas.

## 6. Receita por tipo de reparo

In [10]:
receita = entregues.groupby("tipo_reparo")["valor"].sum().reset_index()
receita["percentual"] = (receita["valor"] / receita["valor"].sum() * 100).round(1)
receita = receita.sort_values("valor")

fig = px.bar(receita, x="valor", y="tipo_reparo", orientation="h",
             title="Receita por tipo de reparo (R$)", text="percentual")
fig.update_traces(texttemplate="%{text}%")
fig.update_layout(xaxis_title="Receita (R$)", yaxis_title="")
fig.show()

print(f"Receita total: R$ {entregues['valor'].sum():,.2f}")
print(f"Ticket médio:  R$ {entregues['valor'].mean():,.2f}")

Receita total: R$ 813,414.35
Ticket médio:  R$ 349.40


**Troca de tela** é o serviço que mais dá dinheiro: sozinha traz mais da metade da receita.
Reinstalação de software é o que menos rende.

## 7. Conclusões

1. **Tela quebrada é o carro-chefe:** 35,6% das OS e **55,9% da receita** (R$ 454 mil de
   R$ 813 mil). Vale garantir estoque de telas para não atrasar esses reparos.
2. **Dezembro e janeiro são o pico:** em média **267 aparelhos/mês**, contra **207** nos meses
   normais (cerca de **29% a mais**). Bom momento para reforçar a equipe.
3. **Danos por água volta muito:** **30,3%** dos aparelhos entregues voltam na garantia, contra
   uma média geral de **10,0%**. Vale rever o procedimento ou avisar o cliente do risco.
4. **O técnico novo precisa de apoio:** Marcos tem **16,6%** de retorno, mais que o dobro de
   Rafael (7,1%) e Juliana (7,2%). Sugestão: treinamento ou revisão dos reparos dele.
5. **Reparo de placa é o mais demorado:** **6,2 dias** em média, contra 0,4 a 2,8 dias nos
   reparos mais simples.